In [1]:
import cv2
import time
import os
import threading
import pickle
import math
import winsound

In [2]:
with open("models/eye_state_model.pkl", "rb") as file:
    model = pickle.load(file)

print("Model loaded successfully")
print(model)
print("Features expected:", model.n_features_in_)

Model loaded successfully
SVC(kernel='linear', probability=True)
Features expected: 4096


In [3]:
IMG_SIZE = int(math.sqrt(model.n_features_in_))

print("IMG_SIZE =", IMG_SIZE)


CLOSED_THRESHOLD_SECONDS = 5

closed_start_time = None
alarm_playing = False
prev_eyes = []

IMG_SIZE = 64


In [4]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

if face_cascade.empty():
    print("Face cascade not loaded")

if eye_cascade.empty():
    print("Eye cascade not loaded")

In [5]:
def play_alarm():
    winsound.Beep(1000, 800)

In [6]:
def safe_play_alarm():
    global alarm_playing

    if alarm_playing:
        return

    alarm_playing = True
    play_alarm()
    alarm_playing = False

In [7]:
def log_alert():
    with open("alert_log.txt", "a") as f:
        f.write(f"Drowsiness alert at {time.ctime()}\n")

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not opened")

closed_start_time = None
prev_eyes = []
alert_saved = False

while True:
    ret, frame = cap.read()

    if not ret:
        print("Camera error")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=6,
        minSize=(100, 100)
    )

    closed_detected = False
    eye_status = "No Eyes"
    closed_duration = 0

    for (x, y, w, h) in faces:

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (255, 0, 0),
            2
        )

        roi_gray = gray[y:y + h // 2, x:x + w]
        roi_color = frame[y:y + h // 2, x:x + w]

        eyes = eye_cascade.detectMultiScale(
            roi_gray,
            scaleFactor=1.05,
            minNeighbors=5,
            minSize=(25, 25),
            maxSize=(100, 100)
        )

        eyes = sorted(
            eyes,
            key=lambda e: e[2] * e[3],
            reverse=True
        )[:2]

        if len(eyes) > 0:
            prev_eyes = eyes
        elif len(prev_eyes) > 0:
            eyes = prev_eyes
        else:
            eyes = []

        closed_count = 0
        total_eyes = 0

        for (ex, ey, ew, eh) in eyes:

            eye_img = roi_gray[ey:ey + eh, ex:ex + ew]

            if eye_img is None or eye_img.size == 0:
                continue

            eye_img = cv2.resize(
                eye_img,
                (IMG_SIZE, IMG_SIZE)
            )

            eye_img = eye_img.flatten().reshape(1, -1)

            prediction = model.predict(eye_img)[0]

            total_eyes += 1

            if prediction == 1:
                closed_count += 1
                color = (0, 0, 255)
            else:
                color = (0, 255, 0)

            cv2.rectangle(
                roi_color,
                (ex, ey),
                (ex + ew, ey + eh),
                color,
                2
            )

        if total_eyes > 0:
            if closed_count >= 1:
                closed_detected = True
                eye_status = "Eyes Closed"
            else:
                eye_status = "Eyes Open"

        break

    if closed_detected:

        if closed_start_time is None:
            closed_start_time = time.time()

        closed_duration = time.time() - closed_start_time

        if closed_duration >= CLOSED_THRESHOLD_SECONDS:

            cv2.putText(
                frame,
                "DROWSINESS ALERT!",
                (40, 130),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.2,
                (0, 0, 255),
                3
            )

            threading.Thread(
                target=safe_play_alarm,
                daemon=True
            ).start()

            if not alert_saved:
                os.makedirs("screenshots", exist_ok=True)

                cv2.imwrite(
                    "screenshots/drowsiness.jpg",
                    frame
                )

                log_alert()
                alert_saved = True

    else:
        closed_start_time = None
        alert_saved = False

    cv2.putText(
        frame,
        eye_status,
        (40, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Closed Time: {closed_duration:.2f}s",
        (40, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.imshow("Driver Drowsiness", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

NameError: name 'model' is not defined

: 